### Tratamento e Estatísticas Básicas dos Dados
- Padroniza nomes de colunas (ES -> PT) e valores
- Converte tempo de quizzes para segundos
- Remove alunos que não fizeram nenhum quiz
- Faz One-Hot Encoding de variáveis categóricas (robusto a versão do sklearn, com fallback para pandas.get_dummies)
- Calcula estatísticas descritivas para colunas numéricas selecionadas
- Remove categorias inválidas oriundas de erro de planilha (ex.: STEM_#ERROR!)

In [10]:
import os
import re
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    pass

Dicionários de renomeação de colunas para padronizar nomes do espanhol para português e tratar erros do XSLS

In [11]:
"""
Cria um d (ASCII, sem acentos nos nomes de colunas para evitar problemas).
"""
rename_map = {
    # Básicos
    "Periodo": "Periodo",
    "Grupo": "Grupo",
    "Horario": "Horario",
    "Tipo_Documento": "Tipo_Documento",
    "Edad": "Idade",
    "Genero": "Genero",
    "STEM": "STEM",
    "MejoraNotaQuices": "MelhoraNotaQuizzes",
    "Calificación_Oficial": "Nota_Oficial",
    "Aprobo": "Aprovou",
    "Nombre_Programa_Academico": "Nome_Programa_Academico",
    "Nombre_Programa_Académico": "Nome_Programa_Academico",
    "Proyecto_Parte1": "Projeto_Parte1",
    "Proyecto_Parte2": "Projeto_Parte2",
    "Talleres": "Oficinas",
    "Quices": "Quizzes",
    "CalcNotaQuiz": "CalcNotaQuiz",
    "Parcial_1": "Parcial_1",
    "Parcial_2": "Parcial_2",
    "Quiz1": "Quiz1",
    "Quiz2": "Quiz2",
    "Quiz3": "Quiz3",
    "Quiz4": "Quiz4",
    "Quiz5": "Quiz5",
    "Quiz6": "Quiz6",
    "Quiz7": "Quiz7",
    "Quiz8": "Quiz8",
    # Às vezes este campo aparece com variação
    "Cuánto mejora?": "Quanto_Melhora",
    "Cuanto mejora?": "Quanto_Melhora",
}

# Renomeação dinâmica de "Fecha_QuizN" -> "Data_QuizN" e "TiempoQN" -> "TempoQN"
for i in range(1, 13):  # cobre até 12, caso o dataset cresça
    rename_map[f"Fecha_Quiz{i}"] = f"Data_Quiz{i}"
    rename_map[f"TiempoQ{i}"] = f"TempoQ{i}"


MAPA_PROGRAMAS = {
    "COMUNICACIÓN SOCIAL": "Comunicacao Social",
    "DERECHO": "Direito",
    "INGENIERÍA CIVIL": "Engenharia Civil",
    "ADMINISTRACIÓN DE NEGOCIOS": "Administracao de Negocios",
    "INGENIERÍA DE DISEÑO DE PRODUCTO": "Engenharia de Design de Produto",
    "MERCADEO": "Marketing",
    "PSICOLOGÍA": "Psicologia",
    "INGENIERÍA FÍSICA": "Engenharia Fisica",
    "NEGOCIOS INTERNACIONALES": "Negocios Internacionais",
    "BIOLOGÍA": "Biologia",
    "CIENCIAS POLÍTICAS": "Ciencias Politicas",
    "ECONOMÍA": "Economia",
    "CONTADURÍA PÚBLICA": "Contabilidade Publica",
    "MÚSICA": "Musica",
    "LITERATURA": "Literatura",
    "INGENIERÍA DE PROCESOS": "Engenharia de Processos",
    "CONVENIO MOVILIDAD PREGRADO (CONVENIOS - MOVILIDAD NACIONAL - ASISTENTES PREGRADO)":
        "Convenio Mobilidade Graduacao (Convenios - Mobilidade Nacional - Assistentes Graduacao)",
}

MAPA_APROV = {
    # valores com e sem acento
    "Aprobó": "Aprovou",
    "Reprobó": "Reprovou",
    "Aprobo": "Aprovou",
    "Reprobo": "Reprovou",
}

MAPA_IDADE = {
    "Mayor": "Maior",
    "Menor": "Menor",
}

MAPA_GENERO = {
    "femenino": "Feminino",
    "masculino": "Masculino",
    "Femenino": "Feminino",
    "Masculino": "Masculino",
}

# Padronização do conteúdo de STEM (mantendo idioma ES para os rótulos finais: "Si"/"No")
MAPA_STEM = {
    "Sí": "Sim",
    "SÍ": "Sim",
    "si": "Sim",
    "SIM": "Sim",
    "Sim": "Sim",
    "YES": "Sim",
    "Yes": "Sim",
    "TRUE": "Sim",
    "True": "Sim",
    "1": "Sim",
    "No": "No",
    "NO": "No",
    "Nao": "No",
    "Não": "No",
    "nao": "No",
    "FALSE": "No",
    "False": "No",
    "0": "No",
}

# Tokens de erro comuns de planilha/Excel que devemos tratar como NaN
ERROR_TOKENS = {
    "#ERROR!", "#DIV/0!", "#N/A", "#NAME?", "#NULL!", "#NUM!", "#VALUE!", "#REF!",
    "N/D", "N/A", "NA", "NaN", "nan", "None", "NONE"
}

Funções para aplicar os mapas no dataset

In [12]:
def replace_excel_errors(df: pd.DataFrame, cols: Optional[List[str]] = None) -> pd.DataFrame:
    """
    Substitui tokens de erro comuns por NaN nas colunas informadas (ou em todas as 'object').
    """
    df = df.copy()
    if cols is None:
        cols = df.select_dtypes(include="object").columns.tolist()
    for c in cols:
        if c in df.columns:
            df[c] = df[c].replace(list(ERROR_TOKENS), np.nan)
    return df


def padroniza_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Renomeia colunas para um padrão comum em PT (ASCII).
    - Substitui tokens de erro de planilha por NaN em colunas textuais.
    - Padroniza valores textuais em colunas-chave (programa, aprovação, idade, gênero, STEM).
    """
    df = df.copy()
    # rename_map = build_rename_map()
    df = df.rename(columns=rename_map)

    # Substituir tokens de erro (somente em colunas de texto)
    df = replace_excel_errors(df)

    # Padronização de valores
    col_prog = "Nome_Programa_Academico"
    if col_prog in df.columns:
        df[col_prog] = (
            df[col_prog]
            .astype(str)
            .str.strip()
            .replace(MAPA_PROGRAMAS, regex=False)
        )

    col_aprov = "Aprovou"
    if col_aprov in df.columns:
        df[col_aprov] = (
            df[col_aprov]
            .astype(str)
            .str.strip()
            .replace(MAPA_APROV, regex=False)
        )

    col_idade = "Idade"
    if col_idade in df.columns:
        df[col_idade] = (
            df[col_idade]
            .astype(str)
            .str.strip()
            .replace(MAPA_IDADE, regex=False)
        )

    col_gen = "Genero"
    if col_gen in df.columns:
        df[col_gen] = (
            df[col_gen]
            .astype(str)
            .str.strip()
            .replace(MAPA_GENERO, regex=False)
        )

    # Padroniza STEM e remove tokens de erro já transformados em NaN
    col_stem = "STEM"
    if col_stem in df.columns:
        mask = df[col_stem].notna()
        if mask.any():
            tmp = df.loc[mask, col_stem].astype(str).str.strip().replace(MAPA_STEM, regex=False)
            df.loc[mask, col_stem] = tmp

    return df

Função para converter tempo de Quizzes para segundos

In [13]:
# Regex pré-compilado para parsing de tempo
_PATTERN_TEMPO = re.compile(
    r"""
    ^\s*
    (?:(\d+)\s*minuto(?:s)?)?   # grupo 1: minutos (opcional)
    (?:\s*)                     # espaço opcional
    (?:(\d+)\s*segundo(?:s)?)?  # grupo 2: segundos (opcional)
    \s*$
    """,
    re.IGNORECASE | re.VERBOSE
)

def parse_to_seconds(val) -> float:
    """
    Converte entradas como "2 minutos 10 segundos", "3:20", "01:02:03", "200" (segundos)
    em número de segundos (int). Retorna np.nan para inválidos.
    """
    if pd.isna(val):
        return np.nan

    # Se já for número (int/float, não bool), tratar como segundos
    if isinstance(val, (int, float)) and not isinstance(val, bool):
        try:
            return int(val)
        except Exception:
            return np.nan

    # Se for string, tentar várias formas
    if isinstance(val, str):
        s = val.strip()
        # Tentar converter diretamente para número
        try:
            return int(float(s))
        except Exception:
            pass

        # Normalizar conectivos comuns
        s_norm = s.replace(" e ", " ").replace(",", " ")

        # Ex: "2 minutos 30 segundos"
        m = _PATTERN_TEMPO.match(s_norm)
        if m:
            mins = m.group(1)
            secs = m.group(2)
            total = 0
            if mins is not None:
                total += int(mins) * 60
            if secs is not None:
                total += int(secs)
            return total if (mins is not None or secs is not None) else np.nan

        # Formatos com dois ou três campos separados por ":"
        if ":" in s_norm:
            parts = [p.strip() for p in s_norm.split(":")]
            if len(parts) == 2 and all(p.isdigit() for p in parts):
                mm, ss = map(int, parts)
                return mm * 60 + ss
            if len(parts) == 3 and all(p.isdigit() for p in parts):
                hh, mm, ss = map(int, parts)
                return hh * 3600 + mm * 60 + ss

    return np.nan


def converte_colunas_tempo(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converte colunas de tempo de quiz para segundos.
    Considera colunas que comecem com 'TempoQ' (já padronizado) ou 'TiempoQ' (caso não tenham sido renomeadas).
    """
    df = df.copy()
    # Identifica colunas de tempo
    cols_tempo = [c for c in df.columns if re.match(r"^(TempoQ|TiempoQ)\d+$", c)]
    for col in cols_tempo:
        df[col] = df[col].apply(parse_to_seconds)
    return df

Função para remover alunos não presentes (que não realizaram nenhum quiz)

In [ ]:
def filtra_alunos_presentes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Mantém apenas alunos que realizaram ao menos um quiz (com base nas colunas de data).
    Remove as colunas de datas de fechamento dos quizzes depois de filtrar.
    """
    df = df.copy()

    cols_data = [c for c in df.columns if re.match(r"^(Data_Quiz|Fecha_Quiz)\d+$", c)]
    if not cols_data:
        # Se não existirem colunas de data, retorna o df como está (sem filtrar)
        return df

    # Filtrar linhas onde pelo menos uma data não é nula
    df_filtrado = df.dropna(subset=cols_data, how="all")
    # Remover as colunas de datas
    df_filtrado = df_filtrado.drop(columns=cols_data)
    return df_filtrado

Funções para tratar variáveis categóricacs

In [ ]:
def fit_onehot_encoder(df_cat: pd.DataFrame, cat_cols: List[str]):
    """
    Tenta usar sklearn OneHotEncoder; se não disponível, cai para pandas.get_dummies.
    Retorna um dicionário com:
      - "type": "sklearn" ou "pandas"
      - "encoder": o encoder do sklearn (se houver)
      - "columns": lista de nomes de colunas resultantes
      - "cat_cols": colunas categóricas usadas
    """
    # Verifica apenas as colunas que existem
    cat_cols_exist = [c for c in cat_cols if c in df_cat.columns]
    if not cat_cols_exist:
        return {"type": "none", "encoder": None, "columns": [], "cat_cols": []}

    X_fit = df_cat[cat_cols_exist].copy()
    # garantir que tokens de erro não entrem como categorias
    X_fit = replace_excel_errors(X_fit, cols=cat_cols_exist)
    # padronizar nulos como 'MISSING'
    X_fit = X_fit.fillna("MISSING").astype(str)

    try:
        from sklearn.preprocessing import OneHotEncoder
        try:
            enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        except TypeError:
            # Compatibilidade com sklearn < 1.2
            enc = OneHotEncoder(sparse=False, handle_unknown="ignore")

        enc.fit(X_fit)
        col_names = list(enc.get_feature_names_out(cat_cols_exist))
        return {"type": "sklearn", "encoder": enc, "columns": col_names, "cat_cols": cat_cols_exist}

    except Exception:
        # Fallback: pandas.get_dummies
        dummies = pd.get_dummies(X_fit, prefix=cat_cols_exist, dummy_na=False)
        return {
            "type": "pandas",
            "encoder": dummies.columns.tolist(),  # usamos nomes de colunas como "modelo" do encoder
            "columns": dummies.columns.tolist(),
            "cat_cols": cat_cols_exist
        }

def aplica_onehot(df: pd.DataFrame, encoder_info) -> pd.DataFrame:
    """
    Aplica o encoder (sklearn ou pandas) e retorna df com colunas categóricas substituídas por dummies.
    Garante que colunas contendo '#ERROR!' não permaneçam no resultado.
    """
    df = df.copy()

    if encoder_info["type"] == "none":
        return df

    cat_cols_exist = encoder_info.get("cat_cols", [])
    # substituir tokens de erro e padronizar nulos
    X = df[cat_cols_exist].copy()
    X = replace_excel_errors(X, cols=cat_cols_exist)
    X = X.fillna("MISSING").astype(str)

    if encoder_info["type"] == "sklearn":
        enc = encoder_info["encoder"]
        arr = enc.transform(X)
        encoded_df = pd.DataFrame(arr, columns=encoder_info["columns"], index=df.index)
        df_out = pd.concat([df.drop(columns=cat_cols_exist), encoded_df], axis=1)

    elif encoder_info["type"] == "pandas":
        dummies_df = pd.get_dummies(X, prefix=cat_cols_exist, dummy_na=False)
        all_cols = encoder_info["columns"]

        # Garante todas as colunas na mesma ordem; colunas faltantes são preenchidas com 0
        for col in all_cols:
            if col not in dummies_df.columns:
                dummies_df[col] = 0
        dummies_df = dummies_df[all_cols]

        df_out = pd.concat([df.drop(columns=cat_cols_exist), dummies_df], axis=1)

    else:
        return df

    # Remover quaisquer colunas vindas de categorias inválidas como '#ERROR!'
    bad_cols = [c for c in df_out.columns if "#ERROR!" in c]
    if bad_cols:
        df_out = df_out.drop(columns=bad_cols)

    return df_out

Função para calcular estatísticas básicas

In [17]:
def estatisticas_basicas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula estatísticas descritivas para colunas numéricas selecionadas:
    - Quiz1..Quiz12 (se existirem)
    - TempoQ1..TempoQ12 (se existirem)
    - Oficinas (ou Talleres), Parcial_1, Parcial_2, CalcNotaQuiz, Nota_Oficial
    Retorna um DataFrame semelhante a df.describe() porém restringindo às colunas de interesse que existirem.
    """
    possiveis = []
    # Quizzes (notas)
    possiveis += [f"Quiz{i}" for i in range(1, 13)]
    # Tempos
    possiveis += [f"TempoQ{i}" for i in range(1, 13)]
    # Outros
    possiveis += ["Oficinas", "Talleres", "Parcial_1", "Parcial_2", "CalcNotaQuiz", "Nota_Oficial"]

    cols_exist = [c for c in possiveis if c in df.columns]
    if not cols_exist:
        return pd.DataFrame()

    # Seleciona apenas numéricas
    num_cols = [c for c in cols_exist if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        return pd.DataFrame()

    return df[num_cols].describe().copy()

Função para executar todas as etapas do processamento

In [18]:
def pipeline_processamento(
    xlsx1: str,
    xlsx2: str,
    sheet: int = 1,
    salvar_dir: Optional[str] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Executa todo o pipeline e retorna:
      - df_consolidado: dados dos dois semestres, depois de padronizar, converter tempos e filtrar presentes
      - df_onehot: df_consolidado com variáveis categóricas codificadas
      - tabela_stats: estatísticas descritivas de colunas numéricas selecionadas

    Se salvar_dir for fornecido, grava CSVs:
      - {salvar_dir}/dados_consolidados.csv
      - {salvar_dir}/dados_onehot.csv
      - {salvar_dir}/estatisticas.csv
    """
    # 1) Carregar
    df1_raw = pd.read_excel(xlsx1, sheet_name=sheet)
    df2_raw = pd.read_excel(xlsx2, sheet_name=sheet)

    # 2) Padronizar colunas/valores e limpar tokens de erro
    df1 = padroniza_df(df1_raw)
    df2 = padroniza_df(df2_raw)

    # 3) Converter colunas de tempo para segundos
    df1 = converte_colunas_tempo(df1)
    df2 = converte_colunas_tempo(df2)

    # 4) Filtrar alunos que não fizeram nenhum quiz (com base em colunas de datas), e remover as colunas de datas
    df1_presentes = filtra_alunos_presentes(df1)
    df2_presentes = filtra_alunos_presentes(df2)

    # 5) Concatenar
    df_consolidado = pd.concat([df1_presentes, df2_presentes], ignore_index=True)

    # 6) Remover linhas com STEM ausente (para evitar categorias inválidas e 'MISSING' em STEM, replicando a lógica original)
    if "STEM" in df_consolidado.columns:
        df_consolidado = df_consolidado.dropna(subset=["STEM"])

    # 7) One-Hot Encoding nas colunas categóricas padronizadas
    cat_cols = ["Tipo_Documento", "Idade", "Genero", "STEM", "MelhoraNotaQuizzes", "Aprovou"]
    encoder_info = fit_onehot_encoder(df_consolidado, cat_cols)
    df_onehot = aplica_onehot(df_consolidado, encoder_info)

    # 8) Estatísticas descritivas
    tabela_stats = estatisticas_basicas(df_consolidado)

    # 9) Salvar, se solicitado
    if salvar_dir:
        os.makedirs(salvar_dir, exist_ok=True)
        df_consolidado.to_csv(os.path.join(salvar_dir, "dados_consolidados.csv"), index=False, encoding="utf-8")
        df_onehot.to_csv(os.path.join(salvar_dir, "dados_onehot.csv"), index=False, encoding="utf-8")
        if not tabela_stats.empty:
            tabela_stats.to_csv(os.path.join(salvar_dir, "estatisticas.csv"), encoding="utf-8")
        else:
            pd.DataFrame().to_csv(os.path.join(salvar_dir, "estatisticas.csv"), index=False, encoding="utf-8")

    return df_consolidado, df_onehot, tabela_stats

Configura camingo das planilhas e Executa o pipeline de processamento

In [19]:
XLSX1_PATH = "../Datos_Anonimo_20231_v2.xlsx"  # 1º semestre
XLSX2_PATH = "../Datos_Anonimo_20232_v2.xlsx"  # 2º semestre
SHEET_INDEX = 1                                # segunda aba
SALVAR_DIR = None                              # por ex.: "./saidas" para salvar CSVs

try:
    df_consolidado, df_onehot, tabela_stats = pipeline_processamento(
        xlsx1=XLSX1_PATH,
        xlsx2=XLSX2_PATH,
        sheet=SHEET_INDEX,
        salvar_dir=SALVAR_DIR
    )

    print("\nResumo:")
    print(f"- Linhas no consolidado: {len(df_consolidado):,}")
    print(f"- Colunas no consolidado: {df_consolidado.shape[1]}")
    print(f"- Colunas no One-Hot: {df_onehot.shape[1]}")
    if not tabela_stats.empty:
        print("- Estatísticas descritivas calculadas para as colunas numéricas selecionadas.")
    else:
        print("- Nenhuma estatística descritiva (não foi possível identificar colunas numéricas selecionadas).")

    # Amostras
    display(df_consolidado.head(3))
    display(df_onehot.head(3))
    if not tabela_stats.empty:
        display(tabela_stats)

except FileNotFoundError as e:
    print("Arquivo não encontrado:", e)
    print("Ajuste as variáveis XLSX1_PATH e XLSX2_PATH e execute a célula novamente.")


Resumo:
- Linhas no consolidado: 1,375
- Colunas no consolidado: 34
- Colunas no One-Hot: 42
- Estatísticas descritivas calculadas para as colunas numéricas selecionadas.


,Periodo,Grupo,Horario,Tipo_Documento,Idade,Genero,Nome_Programa_Academico,STEM,Parcial_1,Parcial_2,...,Quiz6,TempoQ6,Quiz7,TempoQ7,Quiz8,CalcNotaQuiz,MelhoraNotaQuizzes,Quanto_Melhora,Nota_Oficial,Aprovou
0,2023-1,11,Viernes de 03:00PM a 06:00PM,CC,Maior,Feminino,Comunicacao Social,No,4.20,0.0,...,0.00,600,0.0,600,5,1.56250,True,2.73750,2.7,Reprovou
1,2023-1,16,Jueves de 06:00PM a 09:00PM,CC,Maior,Feminino,Direito,No,3.75,5.0,...,2.50,133,0.0,600,5,2.65625,True,2.34375,3.0,Aprovou
2,2023-1,11,Viernes de 03:00PM a 06:00PM,CC,Maior,Masculino,Engenharia Civil,SI,0.00,0.0,...,2.75,105,0.0,600,5,1.28125,True,2.01875,1.6,Reprovou


,Periodo,Grupo,Horario,Nome_Programa_Academico,Parcial_1,Parcial_2,Projeto_Parte1,Projeto_Parte2,Oficinas,Quizzes,...,Idade_Maior,Idade_Menor,Genero_Feminino,Genero_Masculino,STEM_No,STEM_SI,MelhoraNotaQuizzes_False,MelhoraNotaQuizzes_True,Aprovou_Aprovou,Aprovou_Reprovou
0,2023-1,11,Viernes de 03:00PM a 06:00PM,Comunicacao Social,4.20,0.0,0.00,0.00,4.50000,4.3,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
1,2023-1,16,Jueves de 06:00PM a 09:00PM,Direito,3.75,5.0,2.53,2.53,2.08375,5.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
2,2023-1,11,Viernes de 03:00PM a 06:00PM,Engenharia Civil,0.00,0.0,0.00,0.00,3.50000,3.3,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0


,Quiz1,Quiz2,Quiz3,Quiz4,Quiz5,Quiz6,Quiz7,Quiz8,TempoQ1,TempoQ2,TempoQ3,TempoQ4,TempoQ5,TempoQ6,TempoQ7,Oficinas,Parcial_1,Parcial_2,CalcNotaQuiz,Nota_Oficial
count,1375.000000,1375.00000,1375.000000,1375.000000,1375.000000,1375.000000,1375.000000,1375.000000,1375.000000,1375.000000,1374.000000,1375.000000,1375.000000,1375.000000,1375.000000,1375.000000,1375.000000,1375.000000,1375.000000,1375.000000
mean,4.306945,3.98072,4.125455,3.757120,4.245433,3.931956,3.943302,4.174545,181.747636,218.893818,225.923581,240.989818,152.433455,227.927273,244.254545,4.074237,4.221644,4.226727,4.058185,4.238982
std,1.232645,1.29524,1.348448,1.452105,1.625135,1.575427,1.710152,1.856988,121.817034,127.895445,124.040659,127.130954,189.072046,146.739956,165.288414,1.224939,1.169448,1.341164,0.804099,0.812319
min,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,33.000000,24.000000,32.000000,30.000000,2.000000,34.000000,35.000000,0.000000,0.000000,0.000000,0.273750,0.200000
25%,3.750000,3.44000,3.750000,2.810000,4.690000,3.750000,3.750000,5.000000,100.000000,126.000000,137.000000,149.500000,5.000000,118.000000,121.000000,3.600000,3.800000,4.000000,3.603750,3.900000
50%,5.000000,4.38000,5.000000,3.750000,5.000000,5.000000,5.000000,5.000000,151.000000,186.000000,199.000000,219.000000,87.000000,188.000000,195.000000,4.500000,4.800000,5.000000,4.218750,4.500000
75%,5.000000,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,225.000000,285.000000,287.750000,313.000000,218.000000,302.000000,309.000000,5.000000,5.000000,5.000000,4.687500,4.800000
max,5.000000,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,5.000000,5.500000,5.500000,5.000000,5.000000
